# Notebook 07 — Fine-tuning the Cross-Encoder Reranker

**Pipeline position:** optimization experiment for the reranker (assignment section 6, "cross-encoder fine-tuning"). DOCUMENTED NEGATIVE RESULT — not used in production.

1. Generate silver training labels: retrieve top-20 with FT-E5, label each passage by token overlap with the gold answer (>=0.30 = relevant).
2. Fine-tune `seroe/bge-reranker-v2-m3-turkish-triplet` as a binary relevance classifier (AutoModelForSequenceClassification, num_labels=1).

**Why it regressed (-11% F1, p<0.001):** ~52% of training queries had NO positive passage in retrieval, so the silver labels taught the model to rank one irrelevant passage over another. Production therefore uses the off-the-shelf reranker. (Recovered from the original Colab session.)


In [ ]:
############################################################
# RERANKER TRAINING DATA GENERATION
#
# Strategy:
# 1. Take questions from QA datasets
# 2. Retrieve top-20 passages from fine-tuned FAISS
# 3. Auto-label: high token overlap with gold answer → positive
# 4. Hard negatives = retrieved but irrelevant passages
############################################################
import pandas as pd
import numpy as np
import pickle, faiss, re, json
from pathlib import Path
from collections import Counter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
TURKISH_LOWER_MAP = str.maketrans("İIÖÜÇŞĞ", "iıöüçşğ")

def normalize_turkish(text):
    text = text.translate(TURKISH_LOWER_MAP).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def token_overlap(text_a, text_b):
    a = Counter(normalize_turkish(text_a).split())
    b = Counter(normalize_turkish(text_b).split())
    if not a or not b: return 0.0
    common = sum((a & b).values())
    return common / min(sum(a.values()), sum(b.values()))

# Load QA datasets
qa1 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_qa.parquet'))
qa2 = pd.read_parquet(str(DRIVE / 'data' / 'raw' / 'turkish_law_chatbot.parquet'))
qa1 = qa1.rename(columns={'question': 'query', 'answer': 'answer'})
qa2 = qa2.rename(columns={'Soru': 'query', 'Cevap': 'answer'})
qa_all = pd.concat([qa1[['query','answer']], qa2[['query','answer']]], ignore_index=True).dropna().reset_index(drop=True)
print(f"Total QA pairs: {len(qa_all):,}")

# Sample 5000 for reranker training
np.random.seed(42)
qa_sample = qa_all.sample(min(5000, len(qa_all)), random_state=42).reset_index(drop=True)
print(f"Sampled: {len(qa_sample):,}")

# Load fine-tuned FAISS + embedding model
print("Loading FAISS index...")
ft_index = faiss.read_index(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.index'))
with open(str(DRIVE / 'indexes' / 'finetuned' / 'faiss_ft.mapping.pkl'), 'rb') as f:
    ft_ids = pickle.load(f)

# Load chunk texts
import pyarrow.parquet as pq
print("Loading chunk texts...")
pf = pq.ParquetFile(str(DRIVE / 'data' / 'processed' / 'chunks_filtered.parquet'))
id_to_text = {}
for batch in pf.iter_batches(batch_size=100_000, columns=['chunk_id', 'text']):
    df = batch.to_pandas()
    for _, row in df.iterrows():
        id_to_text[row['chunk_id']] = row['text']
    del df
print(f"  Loaded {len(id_to_text):,} chunks")

print("Loading embedding model...")
embed_model = SentenceTransformer(
    str(DRIVE / 'models' / 'e5-checkpoints' / 'checkpoint-10000'),
    device='cuda'
)
print("Ready.")

In [ ]:
############################################################
# Generate (query, passage, label) pairs
# Retrieve top-20 for each query, label by overlap with answer
# label: 1 = relevant (overlap > 0.3), 0 = irrelevant
############################################################
import gc

ft_index.nprobe = 16
OVERLAP_THRESHOLD = 0.3
TOP_K = 20

training_pairs = []
skipped = 0

for i in tqdm(range(len(qa_sample)), desc="Generating pairs"):
    query = qa_sample.iloc[i]['query']
    answer = qa_sample.iloc[i]['answer']

    # Encode query
    q_emb = embed_model.encode(
        [f"query: {query}"], normalize_embeddings=True
    ).astype(np.float32)

    # Search
    scores, indices = ft_index.search(q_emb, TOP_K)

    positives = []
    negatives = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1: continue
        cid = ft_ids[idx]
        text = id_to_text.get(cid, "")
        if not text: continue

        overlap = token_overlap(answer, text)

        if overlap >= OVERLAP_THRESHOLD:
            positives.append(text)
        else:
            negatives.append(text)

    # Add positive pairs
    for p in positives[:3]:  # max 3 positives per query
        training_pairs.append({"query": query, "passage": p, "label": 1})

    # Add hard negative pairs (same count as positives for balance)
    for n in negatives[:max(len(positives), 2)]:
        training_pairs.append({"query": query, "passage": n, "label": 0})

    if not positives:
        skipped += 1

    if (i+1) % 1000 == 0:
        print(f"  {i+1} queries processed, {len(training_pairs)} pairs so far, {skipped} skipped")

    # Memory cleanup every 500
    if (i+1) % 500 == 0:
        gc.collect()

print(f"\nTotal training pairs: {len(training_pairs):,}")
print(f"Queries with no positives (skipped): {skipped}")

labels = [p['label'] for p in training_pairs]
print(f"Positives: {sum(labels):,}, Negatives: {len(labels)-sum(labels):,}")
print(f"Ratio: {sum(labels)/len(labels):.2%} positive")

In [ ]:
############################################################
# Save training data + free embedding model for reranker training
############################################################

# Save to Drive
reranker_data_path = DRIVE / 'data' / 'processed' / 'reranker_training.json'
with open(str(reranker_data_path), 'w', encoding='utf-8') as f:
    json.dump(training_pairs, f, ensure_ascii=False)
print(f"Saved {len(training_pairs):,} pairs to {reranker_data_path}")

# Free embedding model + FAISS from memory
del embed_model, ft_index, id_to_text
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
############################################################
# FINE-TUNE CROSS-ENCODER RERANKER
# Base: seroe/bge-reranker-v2-m3-turkish-triplet (already Turkish-tuned)
# Task: Binary classification (relevant/irrelevant)
############################################################
import json, torch, gc
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from sklearn.model_selection import train_test_split

DRIVE = Path('/content/drive/MyDrive/hukuk-rag')
RERANKER_MODEL = "seroe/bge-reranker-v2-m3-turkish-triplet"
SAVE_DIR = DRIVE / 'models' / 'reranker-finetuned'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Load training data
with open(str(DRIVE / 'data' / 'processed' / 'reranker_training.json'), encoding='utf-8') as f:
    pairs = json.load(f)
print(f"Training pairs: {len(pairs):,}")

# Train/val split
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42, stratify=[p['label'] for p in pairs])
print(f"Train: {len(train_pairs):,}, Val: {len(val_pairs):,}")

# Load model + tokenizer
print(f"\nLoading {RERANKER_MODEL}...")
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL, num_labels=1
)
reranker_model.to('cuda')
print(f"Loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Dataset class
class RerankerDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        encoding = self.tokenizer(
            item['query'], item['passage'],
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(float(item['label']), dtype=torch.float),
        }

train_dataset = RerankerDataset(train_pairs, reranker_tokenizer)
val_dataset = RerankerDataset(val_pairs, reranker_tokenizer)
print(f"Datasets ready. Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
############################################################
# TRAIN RERANKER
############################################################
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

training_args = TrainingArguments(
    output_dir=str(SAVE_DIR / 'checkpoints'),
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    logging_steps=100,
    fp16=True,
    gradient_accumulation_steps=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=reranker_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("Starting reranker training...")
print(f"  Steps per epoch: {len(train_dataset) // (16 * 2)}")
print(f"  Total steps: {len(train_dataset) // (16 * 2) * 2}")
trainer.train()